# Transaction Anomaly Detection with Isolation Forest

### Fraud Detection Aml Analytics — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of fraud detection and anti-money laundering terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Describe how unsupervised anomaly detection works and its relevance to fraud detection and anti-money laundering.
- Implement an anomaly detection pipeline on synthetic data and interpret results in operational terms.
- Evaluate the trade-off between detection sensitivity and false-alarm rate in a banking monitoring context.

**Estimated time:** 35–45 minutes

**Collection context:** Fraud detection, AML compliance, anomaly detection, and financial crime analytics in banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Most transactions are normal. Fraud is rare and often looks "different." Isolation Forest is a powerful tool because it doesn't need to be told what fraud looks like—it just finds things that are lonely or far away from the rest of the data.

**Input data used:** Transaction amount and frequency (number of transactions per hour).

**What we detect:** Anomalous transactions (potential fraud or system errors).

**ML method used:** Isolation Forest (Unsupervised).

**Learning goal:** Understand how to detect fraud without having a labeled training set.

**Primary stakeholders:** fraud analysts, compliance officers, risk managers, and financial crime investigators.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Transaction Data

We create 2,000 normal transactions and 50 "Anomalies" (e.g., extremely high amounts or very high frequency).

In [ ]:
# Normal transactions (mostly small amounts, low frequency)
n_normal = 2000
amt_normal = RNG.normal(50, 15, n_normal)
freq_normal = RNG.normal(2, 0.5, n_normal)

# Anomalous transactions (Outliers)
n_outliers = 50
amt_outliers = RNG.uniform(500, 1000, n_outliers)
freq_outliers = RNG.uniform(10, 20, n_outliers)

X = pd.DataFrame({
    'amount': np.concatenate([amt_normal, amt_outliers]),
    'frequency': np.concatenate([freq_normal, freq_outliers])
})

print(f"Total transactions: {len(X)}")

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

For this workflow, the data prepared in Step 3 is used directly as input features.

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: flag observations beyond a fixed percentile ---
THRESHOLD_PCT = 97
numeric_cols = X.select_dtypes(include='number').columns.tolist()
if 'anomaly_label' in numeric_cols:
    numeric_cols.remove('anomaly_label')
if 'is_anomaly' in numeric_cols:
    numeric_cols.remove('is_anomaly')
thresholds = X[numeric_cols].quantile(THRESHOLD_PCT / 100)
baseline_flags = (X[numeric_cols] > thresholds).any(axis=1)
print(f"Percentile baseline ({THRESHOLD_PCT}th) flags {baseline_flags.sum()} of {len(X)} observations.")
print("Isolation Forest should produce more precise anomaly boundaries.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

The algorithm works by randomly splitting the data. Anomalies take fewer splits to "isolate" because they are far from the crowd.

In [ ]:
# contamination=0.025 means we expect about 2.5% of the data to be outliers
iso_forest = IsolationForest(contamination=0.03, random_state=RANDOM_SEED)
X['anomaly_label'] = iso_forest.fit_predict(X[['amount', 'frequency']])

# Isolation Forest returns -1 for outliers and 1 for inliers
X['is_anomaly'] = X['anomaly_label'].apply(lambda x: 'Anomaly' if x == -1 else 'Normal')

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='amount', y='frequency', hue='is_anomaly', data=X, palette='Set2', alpha=0.6)
plt.title('Isolation Forest: Detecting Transaction Anomalies')
plt.xlabel('Transaction Amount ($)')
plt.ylabel('Transactions per Hour (Frequency)')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Scatter plot titled "Isolation Forest: Detecting Transaction Anomalies" with Transaction Amount ($) on the x-axis and Transactions per Hour (Frequency) on the y-axis. The chart highlights spatial separation or clustering patterns in the data.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Why this worked:**
- **Isolation:** The algorithm "isolated" the red points because they were different from the dense blue cluster.
- **No Labels:** We didn't need to label any data as "Fraud" to find these outliers.

**Banking Context:** Anomaly detection is the first line of defense. Any transaction flagged as an "Anomaly" would be sent to a human investigator or trigger a multi-factor authentication (MFA) request.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score new observations ---
print("Operational demonstration — anomaly scoring:\n")
new_obs = pd.DataFrame({
    X.columns[0]: [X[X.columns[0]].median(), X[X.columns[0]].quantile(0.99)],
    X.columns[1]: [X[X.columns[1]].median(), X[X.columns[1]].quantile(0.99)],
}, index=["routine", "suspicious"])

scores = iso_forest.decision_function(new_obs)
preds  = iso_forest.predict(new_obs)
for name, sc, pr in zip(new_obs.index, scores, preds):
    status = "FLAGGED" if pr == -1 else "normal"
    print(f"  {name}: anomaly score = {sc:.3f}  {status}")

print("\nFlagged items would be routed to a human investigator.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end fraud detection and anti-money laundering workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- False positives can freeze legitimate accounts, causing financial hardship and customer distrust.
- Fraud models risk encoding biases against specific demographic groups or geographic regions.
- Transaction monitoring must balance fraud prevention with customer privacy rights.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in fraud detection and anti-money laundering?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.